In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/mmb0-1b_smb1-1b_1p/harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad')
adata2 = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/mmb0-1b_smb1-1b_1p/harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad')

adata
adata.obs.head(1)
adata2
adata2.obs.head(1)
sc.pl.spatial(adata, spot_size=20)
sc.pl.spatial(adata2, spot_size=100)

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_110m/07112025_054432_709/adata_ep3_5k_complete_processed.h5ad')

# xhs1000-39b_1p Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 02.09.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import logging
import math
import os
import re
import warnings
import pickle
import random
from collections import defaultdict
from datetime import datetime
from itertools import chain, compress
from pathlib import Path

import anndata as ad
import celltypist
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
import squidpy as sq
from datasets import concatenate_datasets, load_from_disk
from scipy.sparse import csr_matrix
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from app.infer import get_gene_embed, get_average_gene_embed, infer
from app.train import train
from nichejepa.datasets import prepare_dataset
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.embedding import collect_adata_from_folder
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [ ]:
# Set params
dataset = 'xhs1000-39b_1p'
tokenizer = 'cell_graph'
# load_timestamp = '03072025_001818_175'
# load_timestamp = '11062025_183320_055'
#load_timestamp = '27102025_054302_983'
load_timestamp = '24112025_212840_818'
model_type = 'test'
read_checkpoint = 'nichejepa-ep180.pth.tar' # 'nichejepa-ep3_0.pth.tar', 'nichejepa-ep10_0.pth.tar'
aws_run = False

emb_layer = None
n_layer = 3
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12
n_pcs = 0

layer = 3

In [ ]:
# Define paths
model_config_folder_path = '/nfs/team361/sb75/nichejepa/configs/model' # '/nfs/team361/sb75/nichejepa/configs/model' # '/lustre/scratch126/cellgen/team361/sb75/nichejepa/configs/model'
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/lustre/scratch126/cellgen/lotfollahi/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'
model_folder_path = f'/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/{load_timestamp}'
raw_data_folder_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset}'

# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

In [ ]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [ ]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{model_type}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

In [ ]:
if aws_run:
    if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
        params['data']['precomputed_epoch_n_nonzero_tokens'] = params['data']['precomputed_epoch_n_nonzero_tokens'].replace(
            f'/fsx-shared/s3/{dataset}', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer'
        )
    elif 'precomputed_n_nonzero_tokens' in params['data'].keys():
        params['data']['precomputed_n_nonzero_tokens'] = params['data']['precomputed_n_nonzero_tokens'].replace(
            f'/fsx-shared/s3/{dataset}', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer'
        )
    if 'precomputed_epoch_splits' in params['data'].keys():
        params['data']['precomputed_epoch_splits'] = params['data']['precomputed_epoch_splits'].replace(
            f'/fsx-shared/s3/{dataset}', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer'
        )
    elif 'precomputed_split' in params['data'].keys():
        params['data']['precomputed_split'] = params['data']['precomputed_split'].replace(
            f'/fsx-shared/s3/{dataset}', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer'
        )
    params['data']['token_dict_folder_path'] = params['data']['token_dict_folder_path'].replace(
        '/fsx-shared/s3', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer'
    )
    params['data']['tokenized_data_folder_path'] = params['data']['tokenized_data_folder_path'].replace(
        #'/fsx-shared/s3', '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer'
        '/fsx-shared/s3', '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer'
    ).replace(
        'hst_corpus_80m', dataset
    )    
    del(params['data']['test_batch_ids'])
    del(params['data']['val_batch_ids'])

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

## 1.3 Run Notebook Setup

In [ ]:
os.makedirs(f'{folder_path}/figures', exist_ok=True)

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
#sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [ ]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      agg_type='avg',
                      masked_tokens=None,
                      agg_excluded_tokens=None,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[])

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    for obs_col in obs_cols:
        adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_embedding_similarity(adata,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [ ]:
def compute_neighborhood_umap(adata,
                              write_to_disk=True):
    if 'neighborhood_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) neighborhood gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'neighborhood_emb_layer_{layer}',
                        key_added='neighborhood')
    if 'neighborhood_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='neighborhood',
                   key_added='neighborhood_umap')
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    if 'cell_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) cell gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'cell_emb_layer_{layer}',
                        key_added='cell')
    if 'cell_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='cell',
                   key_added='cell_umap')

        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt

from analysis_utils import *

In [ ]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell':
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood':
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            rsc.tl.leiden(adata,
                         neighbors_key=emb_key,
                         key_added=latent_cluster_key,
                         resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        #metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
        #                  index=False)
        
        # Plot UMAPs
        #os.makedirs(f'{folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(custom_palette if label == latent_cluster_key else None),
                           show=False)
                plt.show()
                #plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                #            dpi=300,
                #            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
                         )
            #os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
            #          f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        #if write_to_disk:
            # Save adata
            #adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata

# 2. Analysis

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
params['data']['batch_size'] = 128

In [ ]:
params['data']['precomputed_n_nonzero_tokens'] = f"/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params['data']['tokenized_data_folder_path'].split('/')[-1].split('.dataset')[0]}_n_nonzero_tokens_train.pkl"

In [ ]:
# Load data
train_dataset, _, test_dataset = prepare_dataset(params)
train_dataset

In [ ]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=random.randint(40000, 50000))
logger.info(f'Running... (rank: {rank}/{world_size})')

## 2.1 Benchmarking Test Sets

### 2.1.2 xhs1000-39b_1p (Xenium Human Skin Eczema)

#### 2.1.2.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1000_batch11',
    '1000_batch19',
    '1000_batch32']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhs1000-39b_1p'
niche_key = 'niche_type'
batch_key = 'sample'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * 3

#### 2.1.2.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.2.3 Plot Embedding Similarity Stats

In [ ]:
niche_key = 'niche_type'

In [ ]:
import scanpy as sc
import rapids_singlecell as rsc

In [ ]:
# not too bad - adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/xhs1000-39b_1p/18112025_041246_489/adata_ep29_xhs1000-39b_1p.h5ad')
# adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/xhs1000-39b_1p/18112025_102544_534/adata_ep20_xhs1000-39b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_082427_561/adata_ep25_mmb0-1b_smb1-1b_1p.h5ad') # unrestricted special attention
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_083037_451/adata_ep25_mmb0-1b_smb1-1b_1p.h5ad') # default
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_083419_863/adata_ep25_mmb0-1b_smb1-1b_1p.h5ad') # 1 target
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_084348_215/adata_ep25_mmb0-1b_smb1-1b_1p.h5ad') # higher batch size

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_103608_534/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_113334_369/adata_ep90_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_113632_703/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_113955_366/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_114514_874/adata_ep90_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_114241_653/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_120517_861/adata_ep20_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_215348_528/adata_ep90_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/25112025_215148_295/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_004147_759/adata_ep8 0_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_082347_895/adata_ep75_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_085920_795/adata_ep45_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_091749_638/adata_ep70_mmb0-1b_smb1-1b_1p.h5ad') # very good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_091626_581/adata_ep20_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_085108_608/adata_ep45_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_083112_851/adata_ep45_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_090918_515/adata_ep20_mmb0-1b_smb1-1b_1p.h5ad')


#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_091354_350/adata_ep25_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_092058_174/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad') # good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_091850_760/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad') # not integrated
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_085635_171/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad') # bad
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_091749_638/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad') # good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_085309_386/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad') # no integration

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_212151_442/adata_ep0_mmb0-1b_smb1-1b_1p.h5ad')

# rebuild branch
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225506_151/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_224043_631/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_224601_895/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_224746_032/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_224859_135/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_224943_811/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225042_179/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225144_497/adata_ep100_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/27112025_090852_632/adata_ep5_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225241_772/adata_ep95_mmb0-1b_smb1-1b_1p.h5ad') # very good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225426_344/adata_ep95_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_225701_280/adata_ep30_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/26112025_231211_468/adata_ep5_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_072818_219/adata_ep9_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_072818_219/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_073235_418/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_073612_622/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_073736_021/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/28112025_073851_081/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad')

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/29112025_084225_624/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good with sampling
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/29112025_090153_888/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/30112025_224307_059/adata_ep49_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/01122025_084911_302/adata_ep24_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/01122025_141724_259/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good with good cosine sim
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/01122025_084911_302/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad')
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_125918_190/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # 128 batch size, decent
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_100342_066/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # 64 batch, decent
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_081044_643/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # high cov reg NOPE!
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_081017_728/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # high var reg NOPE!
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_080854_428/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good 0.8 0.2
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_080819_700/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # not good 0.8 0.05
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_075057_618/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # not good 1.0 0.05
#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/02122025_080854_428/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # very good 0.8 0.2

#adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/04122025_102811_334/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # good dev code
adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/08122025_214212_950/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # publication grade

adata = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/mmb0-1b_smb1-1b_1p/17122025_084610_456/adata_ep99_mmb0-1b_smb1-1b_1p.h5ad') # better combined


In [ ]:
adata

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
n_neighbors = 15
emb_key = 'neighborhood_emb'
emb_layer = f'{emb_key}_layer_3'
latent_leiden_resolution = 0.6
latent_cluster_key = f"{emb_key}_clusters_res_{latent_leiden_resolution}"

In [ ]:
rsc.pp.neighbors(
    adata,
    n_neighbors=n_neighbors,
    use_rep=emb_layer,
    n_pcs=None,
    key_added=emb_key)

rsc.tl.umap(adata,
            neighbors_key=emb_key,
            key_added=f'{emb_key}_umap')

In [ ]:
rsc.tl.leiden(adata,
              neighbors_key=emb_key,
              key_added=latent_cluster_key,
              resolution=latent_leiden_resolution)

#### 2.1.2.4 Identify Niches & Cell Types

In [ ]:
adata

In [ ]:
obs_col = 'nichecompass_latent_cluster'
adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']

In [ ]:
spot_sizes = [100, 20, 20]

In [ ]:
import sys
sys.path.append("../../utils")
from analysis_utils import *

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot UMAPs
#os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', obs_col, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    if len(np.unique(adata.obs[label])) > 1:
        sc.pl.umap(adata,
                   neighbors_key=emb_key,
                   title=(emb_key + "_emb_" + label if emb_key not in label else label),
                   color=label,
                   palette=(custom_palette if label == latent_cluster_key else None),
                   show=False)
        plt.show()
        #plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
        #            dpi=300,
        #            bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
                 )
    #os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
    #          f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))

#if write_to_disk:
    # Save adata
    #adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')


In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
labels = adata.obs[obs_col]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})

In [ ]:
import numpy as np

X = adata.obsm['neighborhood_emb_layer_3']  # shape (N, D)

# Normalize rows
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)

# Compute and normalize mean vector
mean_vec = X.mean(axis=0, keepdims=True)
mean_norm = mean_vec / np.linalg.norm(mean_vec)

# Row-wise cosine similarity
cos_sim = (X_norm * mean_norm).sum(axis=1)

# Average
mean_cos = cos_sim.mean()

print(mean_cos)